<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/drive_to_gcs_sync.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://drive.google.com/drive/folders/1sj-RqD5HRypGx-ZLqXpvyzY1CN84qJu-?usp=drive_link" target="_parent"><img src="https://img.shields.io/badge/PyBlender_Render_Farm-blue?logo=googledrive&logoColor=white" alt="PyBlender_Render_Farm"/></a>

# ☁️ Drive ↔ GCS Sync (plyFormat)

**Perfect sync** of the `PointCloud/plyFormat/` folder from Google Drive to GCS.

- ✅ Uploads new files from Drive to GCS
- 🗑️ Deletes files from GCS that no longer exist on Drive
- ⏭️ Skips files that already exist on both

> Uses a **service account key** stored in **Colab Secrets** for GCS auth.
>
> To add the secret: click 🔑 icon in left sidebar → **+ Add new secret** → name it `GCS_SERVICE_ACCOUNT_KEY` → paste the full JSON content of your service account key.

In [ ]:
#@title 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Install GCS SDK
!pip install google-cloud-storage -q

In [ ]:
#@title 3. Authenticate GCS (Service Account via Colab Secrets)
import os
from google.colab import userdata

gcs_key_json = userdata.get('GCS_SERVICE_ACCOUNT_KEY')

GCS_KEY_PATH = "/tmp/gcs_service_account.json"
with open(GCS_KEY_PATH, "w") as f:
    f.write(gcs_key_json)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCS_KEY_PATH
print(f"✅ Service account key loaded from Colab Secrets")
print(f"   Saved to {GCS_KEY_PATH}")

from google.cloud import storage
client = storage.Client()
bucket = client.bucket("pyblender-render-farm")
try:
    next(bucket.list_blobs(max_results=1), None)
    print(f"✅ Connected to gs://pyblender-render-farm")
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
#@title 4. Sync Config

SYNC_CONFIG = {
    # --- Source: plyFormat folder on Google Drive ---
    "DRIVE_SYNC_DIR": "/content/drive/MyDrive/PyBlender_Render_Farm/PointCloud/plyFormat",

    # --- Destination: GCS prefix (mirrors Drive structure) ---
    "GCS_PREFIX": "PointCloud/plyFormat",

    # --- GCS Bucket ---
    "GCS_BUCKET": "pyblender-render-farm",
}

print(f"Drive:  {SYNC_CONFIG['DRIVE_SYNC_DIR']}")
print(f"GCS:    gs://{SYNC_CONFIG['GCS_BUCKET']}/{SYNC_CONFIG['GCS_PREFIX']}/")

import os
src = SYNC_CONFIG["DRIVE_SYNC_DIR"]
if os.path.exists(src):
    total = sum(len(files) for _, _, files in os.walk(src))
    print(f"\nFound {total} files on Drive:")
    for sd in sorted(os.listdir(src)):
        sd_path = os.path.join(src, sd)
        if os.path.isdir(sd_path):
            count = len([f for f in os.listdir(sd_path) if os.path.isfile(os.path.join(sd_path, f))])
            print(f"  {sd}/ ({count} files)")
else:
    print(f"\n❌ Directory not found: {src}")

In [ ]:
#@title 5. Run Perfect Sync (Drive → GCS + delete stale GCS files)
import os
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(SYNC_CONFIG["GCS_BUCKET"])

drive_root = SYNC_CONFIG["DRIVE_SYNC_DIR"]
gcs_prefix = SYNC_CONFIG["GCS_PREFIX"]

# ==========================================
# Step 1: Build set of all Drive files
# ==========================================
drive_files = set()
for root, dirs, files in os.walk(drive_root):
    for fname in files:
        local_path = os.path.join(root, fname)
        relative = os.path.relpath(local_path, drive_root)
        gcs_path = f"{gcs_prefix}/{relative}"
        drive_files.add(gcs_path)

print(f"Drive files: {len(drive_files)}")

# ==========================================
# Step 2: Build set of all GCS files
# ==========================================
print(f"Scanning GCS files under {gcs_prefix}/...")
gcs_blobs = {}
for blob in bucket.list_blobs(prefix=gcs_prefix + "/"):
    if not blob.name.endswith("/"):  # skip directory markers
        gcs_blobs[blob.name] = blob

print(f"GCS files:   {len(gcs_blobs)}")

# ==========================================
# Step 3: Upload new files (Drive → GCS)
# ==========================================
to_upload = drive_files - set(gcs_blobs.keys())
print(f"\n--- Uploading {len(to_upload)} new files ---")

uploaded = 0
for gcs_path in sorted(to_upload):
    relative = os.path.relpath(gcs_path, gcs_prefix)
    local_path = os.path.join(drive_root, relative)
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f"  ✅ {gcs_path} ({size_mb:.1f} MB)")
    uploaded += 1

# ==========================================
# Step 4: Delete stale GCS files (not on Drive)
# ==========================================
to_delete = set(gcs_blobs.keys()) - drive_files
print(f"\n--- Deleting {len(to_delete)} stale GCS files ---")

deleted = 0
for gcs_path in sorted(to_delete):
    gcs_blobs[gcs_path].delete()
    print(f"  🗑️  {gcs_path}")
    deleted += 1

skipped = len(drive_files) - uploaded

print(f"\n{'='*50}")
print(f"  PERFECT SYNC COMPLETE")
print(f"  Uploaded:  {uploaded}")
print(f"  Deleted:   {deleted}")
print(f"  Unchanged: {skipped}")
print(f"{'='*50}")

In [ ]:
#@title 6. Verify: List all files on GCS
gcs_prefix = SYNC_CONFIG["GCS_PREFIX"]
blobs = list(bucket.list_blobs(prefix=gcs_prefix + "/"))

from collections import defaultdict
folders = defaultdict(list)
for b in blobs:
    if b.name.endswith("/"):
        continue
    parts = b.name.split("/")
    folder_key = "/".join(parts[:-1])
    folders[folder_key].append(b)

print(f"gs://{SYNC_CONFIG['GCS_BUCKET']}/{gcs_prefix}/\n")
for folder in sorted(folders.keys()):
    files = folders[folder]
    total_mb = sum((b.size or 0) for b in files) / (1024 * 1024)
    print(f"  {folder}/ — {len(files)} files ({total_mb:.1f} MB)")

total_files = sum(len(v) for v in folders.values())
print(f"\nTotal: {total_files} files")

In [ ]:
#@title 7. Sync GCS RenderImages → Drive
import os
from google.cloud import storage

# ==========================================
# CONFIG: GCS → Drive
# ==========================================
GCS_RENDER_PREFIX = "RenderImages"  # GCS folder to download from
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/PyBlender_Render_Farm/Kaggle_Output"  # Drive destination

client = storage.Client()
bucket = client.bucket("pyblender-render-farm")

blobs = list(bucket.list_blobs(prefix=GCS_RENDER_PREFIX + "/"))
print(f"Found {len(blobs)} files on GCS under {GCS_RENDER_PREFIX}/\n")

downloaded = 0
skipped = 0

for blob in blobs:
    if blob.name.endswith("/"):
        continue

    relative = os.path.relpath(blob.name, GCS_RENDER_PREFIX)
    dest_path = os.path.join(DRIVE_OUTPUT_DIR, relative)

    if os.path.exists(dest_path):
        skipped += 1
        continue

    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    blob.download_to_filename(dest_path)
    downloaded += 1
    print(f"  ✅ {relative}")

print(f"\n{'='*50}")
print(f"  GCS → DRIVE SYNC COMPLETE")
print(f"  Downloaded: {downloaded}")
print(f"  Skipped:    {skipped}")
print(f"  Dest:       {DRIVE_OUTPUT_DIR}")
print(f"{'='*50}")
